# 무료 Photo AI 서버
상단의 **런타임 → 모두 실행**을 누르세요. 마지막에 나오는 `https://…gradio.live` 주소를 웹 앱에 붙여 넣으면 됩니다. 무료 Colab GPU는 사용 가능 여부와 시간이 보장되지 않습니다.

In [ ]:
!pip -q install gradio==5.49.1 transformers==4.56.2 pillow==11.3.0 opencv-python-headless==4.12.0.88


In [ ]:
import json, tempfile, os
from functools import lru_cache
import cv2, numpy as np, torch, gradio as gr
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

def auto_enhance(img):
    rgb = np.array(ImageOps.autocontrast(img, cutoff=0.5))
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8,8)).apply(l)
    out = Image.fromarray(cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2RGB))
    out = ImageEnhance.Color(out).enhance(1.08)
    return out.filter(ImageFilter.UnsharpMask(radius=1.4, percent=135, threshold=3))

def denoise(img):
    bgr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    out = cv2.fastNlMeansDenoisingColored(bgr, None, 7, 7, 7, 21)
    return Image.fromarray(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))

def old_photo(img):
    out = auto_enhance(denoise(img))
    return ImageEnhance.Color(out).enhance(1.18)

def face_restore(img):
    arr = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    gray = cv2.cvtColor(arr, cv2.COLOR_BGR2GRAY)
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    for x,y,w,h in cascade.detectMultiScale(gray, 1.12, 5, minSize=(40,40)):
        pad = int(max(w,h)*.12); x0=max(0,x-pad); y0=max(0,y-pad)
        x1=min(arr.shape[1],x+w+pad); y1=min(arr.shape[0],y+h+pad)
        roi = arr[y0:y1,x0:x1]
        blur = cv2.GaussianBlur(roi,(0,0),1.1)
        arr[y0:y1,x0:x1] = cv2.addWeighted(roi,1.75,blur,-.75,0)
    return Image.fromarray(cv2.cvtColor(arr, cv2.COLOR_BGR2RGB))

@lru_cache(maxsize=1)
def ai_model():
    from transformers import AutoImageProcessor, Swin2SRForImageSuperResolution
    name='caidas/swin2SR-classical-sr-x4-64'
    device='cuda' if torch.cuda.is_available() else 'cpu'
    return AutoImageProcessor.from_pretrained(name), Swin2SRForImageSuperResolution.from_pretrained(name).to(device).eval(), device

def upscale(img, scale):
    processor, model, device = ai_model()
    original=(img.width*scale,img.height*scale)
    inputs=processor(images=img,return_tensors='pt').to(device)
    with torch.inference_mode(): out=model(**inputs).reconstruction.data.squeeze().float().cpu().clamp_(0,1).numpy()
    out=np.moveaxis(out,0,-1); result=Image.fromarray((out*255).round().astype('uint8'))
    return result.resize(original,Image.Resampling.LANCZOS)

def enhance(image_path, options_json):
    opts=json.loads(options_json or '{}'); img=ImageOps.exif_transpose(Image.open(image_path)).convert('RGB'); stages=[]
    if opts.get('auto'): img=auto_enhance(img); stages.append('auto')
    if opts.get('old_photo'): img=old_photo(img); stages.append('old_photo')
    if opts.get('denoise'): img=denoise(img); stages.append('denoise')
    if opts.get('face'): img=face_restore(img); stages.append('face')
    if opts.get('sharpen'): img=img.filter(ImageFilter.UnsharpMask(2.0,190,2)); stages.append('sharpen')
    scale=int(opts.get('scale',1))
    if scale in (2,4): img=upscale(img,scale); stages.append(f'scale{scale}')
    path=os.path.join(tempfile.mkdtemp(prefix='free-photo-ai-'),'enhanced.png'); img.save(path,optimize=True)
    meta={'status':'complete','stages':stages,'width':img.width,'height':img.height,'temporary':True}
    return path,json.dumps(meta,ensure_ascii=False)

with gr.Blocks(title='무료 Photo AI 서버') as demo:
    gr.Markdown('## 무료 Photo AI 서버\n이 탭을 닫으면 서버가 종료됩니다.')
    src=gr.Image(type='filepath',label='테스트 사진'); opts=gr.Textbox(value='{\"auto\":true,\"scale\":1}',visible=False)
    run=gr.Button('테스트 처리'); out=gr.File(); meta=gr.Textbox()
    run.click(enhance,[src,opts],[out,meta],api_name='enhance')
demo.queue().launch(share=True,debug=True)
